# Apartment Return Prediction Experiment

이 노트북은 `apartment.csv`를 불러와 전처리, LightGBM 학습, train/valid/test 평가를 순서대로 실행한다.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

CONFIG_PATH = PROJECT_DIR / 'config' / 'config.yaml'
CONFIG_PATH

## 1. Config와 데이터 로드

In [ ]:
from ml_project.utils import load_yaml
from ml_project.preprocessing import load_raw_data, build_model_frame, split_dataset, log_feature_preview

config = load_yaml(CONFIG_PATH)
raw = load_raw_data(config)
raw.shape, raw.head(2)

## 2. 모델 입력 데이터 생성

In [ ]:
model_df = build_model_frame(raw, config)
bundle = split_dataset(model_df, config)
log_feature_preview(bundle)
{'train': bundle.train.shape, 'valid': bundle.valid.shape, 'test': bundle.test.shape}

## 3. 타깃 분포 확인

In [ ]:
import matplotlib.pyplot as plt
target = config['data']['target_column']
display(bundle.train[target].describe())
bundle.train[target].hist(bins=80)
plt.title(f'Train target distribution: {target}')
plt.show()

## 4. 모델 정의와 학습

In [ ]:
from ml_project.models import build_model

model = build_model(config)
X_train = bundle.train[bundle.feature_columns]
y_train = bundle.train[bundle.target_column]
model.fit(X_train, y_train, categorical_columns=bundle.categorical_columns)
model

## 5. Train / Valid / Test 평가

In [ ]:
from ml_project.utils import evaluate_metrics
import pandas as pd

metrics = config['metrics']['enabled']
rows = []
predictions = {}
for split_name, split_df in [('train', bundle.train), ('valid', bundle.valid), ('test', bundle.test)]:
    if split_df.empty:
        continue
    y_true = split_df[bundle.target_column].to_numpy()
    y_pred = model.predict(split_df[bundle.feature_columns])
    predictions[split_name] = y_pred
    row = {'split': split_name}
    row.update(evaluate_metrics(y_true, y_pred, metrics))
    rows.append(row)

pd.DataFrame(rows)

## 6. Test 예측 샘플

In [ ]:
test_preview = bundle.test[bundle.id_columns + [bundle.target_column]].copy()
if 'test' in predictions:
    test_preview['prediction'] = predictions['test']
test_preview.head(20)

## 7. 모듈형 학습 코드 실행

아래 셀은 같은 과정을 `train.py` 엔트리포인트로 실행한다.

In [ ]:
from ml_project.train import train

result = train(CONFIG_PATH)
result